In [0]:
# ═══════════════════════════════════════════════════════════════════
# NOTEBOOK 7, CELL 1: IMPORTS + LOAD SILVER DATA
# This notebook runs AFTER all training notebooks (3-6).
# It loads the cleaned Silver data and all trained models,
# then produces 6 Gold CSVs for Tableau dashboards.
# ═══════════════════════════════════════════════════════════════════

import time
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.ml.evaluation import (
    RegressionEvaluator,
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
    ClusteringEvaluator
)
from pyspark.ml.regression import (
    LinearRegressionModel, DecisionTreeRegressionModel,
    RandomForestRegressionModel, GBTRegressionModel
)
from pyspark.ml.classification import (
    LogisticRegressionModel, DecisionTreeClassificationModel,
    RandomForestClassificationModel, GBTClassificationModel,
    LinearSVCModel
)
from pyspark.ml.clustering import KMeansModel, BisectingKMeansModel, GaussianMixtureModel
from pyspark.ml import PipelineModel

# ── Load the Silver-layer data (already cleaned + features engineered) ──
df = spark.read.parquet("/Volumes/workspace/default/fema_claims_project/fema_silver")
print(f"✅ Silver data loaded: {df.count():,} rows, {len(df.columns)} columns")

# ── Recreate the train/val/test split using the SAME seed ──
train_df, val_df, test_df = df.randomSplit([0.70, 0.15, 0.15], seed=42)
print(f"Train: {train_df.count():,} | Val: {val_df.count():,} | Test: {test_df.count():,}")

✅ Silver data loaded: 2,641,236 rows, 86 columns
Train: 1,849,405 | Val: 395,896 | Test: 395,935


In [0]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2: LOAD ALL 14 MODELS
# Each model was saved in Notebooks 3/4/5 using .write().overwrite().save()
# If you used MLflow, replace with mlflow.spark.load_model("runs:/<id>/model")
# ═══════════════════════════════════════════════════════════════════

models = {}
model_paths = {
    # Regression models (Notebook 3)
    "Linear_Regression" : "dbfs:/Volumes/workspace/default/fema_claims_project/gold/models/linear_regression",
    "Decision_Tree_Reg":   "dbfs:/Volumes/workspace/default/fema_claims_project/gold/models/dt_regression",
    "Random_Forest_Reg":   "dbfs:/Volumes/workspace/default/fema_claims_project/gold/models/rf_regression",
    "GBT_Reg":             "dbfs:/Volumes/workspace/default/fema_claims_project/gold/models/gbt_regression",
    "Elastic_Net":         "dbfs:/Volumes/workspace/default/fema_claims_project/gold/models/elastic_net",
    # Classification models (Notebook 4)
    "Logistic_Regression": "dbfs:/Volumes/workspace/default/fema_claims_project/gold/models/logistic_regression",
    "Decision_Tree_Cls":   "dbfs:/Volumes/workspace/default/fema_claims_project/gold/models/dt_classification",
    "Random_Forest_Cls":   "dbfs:/Volumes/workspace/default/fema_claims_project/gold/models/rf_classification",
    "GBT_Cls":             "dbfs:/Volumes/workspace/default/fema_claims_project/gold/models/gbt_classification",
    "Linear_SVC":          "dbfs:/Volumes/workspace/default/fema_claims_project/gold/models/linear_svc",
    # Clustering models (Notebook 5)
    "KMeans":              "dbfs:/Volumes/workspace/default/fema_claims_project/gold/models/kmeans",
    "Bisecting_KMeans":    "dbfs:/Volumes/workspace/default/fema_claims_project/gold/models/bisecting_kmeans",
    "GMM":                 "dbfs:/Volumes/workspace/default/fema_claims_project/gold/models/gmm",
    "PCA_KMeans":          "dbfs:/Volumes/workspace/default/fema_claims_project/gold/models/pca_kmeans",
}

for name, path in model_paths.items():
    try:
        models[name] = PipelineModel.load(path)
        print(f"✅ Loaded: {name}")
    except Exception as e:
        print(f"⚠️ Could not load {name}: {e}")

print(f"\n✅ Total models loaded: {len(models)}")

⚠️ Could not load Linear_Regression: [PATH_NOT_FOUND] Path does not exist: dbfs:/Volumes/workspace/default/fema_claims_project/gold/models/linear_regression/metadata. SQLSTATE: 42K03

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.dataPathNotExistError(QueryCompilationErrors.scala:2755)
	at org.apache.spark.sql.execution.datasources.DataSource$.$anonfun$checkAndGlobPathIfNecessary$1(DataSource.scala:1126)
	at scala.collection.immutable.List.flatMap(List.scala:294)
	at scala.collection.immutable.List.flatMap(List.scala:79)
	at org.apache.spark.sql.execution.datasources.DataSource$.checkAndGlobPathIfNecessary(DataSource.scala:1107)
	at org.apache.spark.sql.execution.datasources.DataSource.checkAndGlobPathIfNecessary(DataSource.scala:722)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:543)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$ana

In [0]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3: EVALUATE ALL 14 MODELS → tableau_metrics.csv
# Columns: model, metric, value, category, tuned
# This matches the exact CSV structure from the Tableau tutorial.
# ═══════════════════════════════════════════════════════════════════

# ── Evaluators ──
reg_eval = RegressionEvaluator(labelCol="amountPaidOnBuildingClaim",
                                predictionCol="prediction")
bin_eval = BinaryClassificationEvaluator(labelCol="high_payout")
mc_eval  = MulticlassClassificationEvaluator(labelCol="high_payout",
                                              predictionCol="prediction")
clu_eval = ClusteringEvaluator(predictionCol="prediction",
                               featuresCol="features")

# ── Define which models belong to which category ──
regression_models = ["Linear_Regression", "Decision_Tree_Reg",
                      "Random_Forest_Reg", "GBT_Reg", "Elastic_Net"]
classification_models = ["Logistic_Regression", "Decision_Tree_Cls",
                          "Random_Forest_Cls", "GBT_Cls", "Linear_SVC"]
clustering_models = ["KMeans", "Bisecting_KMeans", "GMM", "PCA_KMeans"]

all_metrics = []  # will become rows in our CSV

# ── REGRESSION EVALUATION ──
for name in regression_models:
    if name not in models:
        continue
    preds = models[name].transform(test_df)
    rmse = reg_eval.evaluate(preds, {reg_eval.metricName: "rmse"})
    r2   = reg_eval.evaluate(preds, {reg_eval.metricName: "r2"})
    mae  = reg_eval.evaluate(preds, {reg_eval.metricName: "mae"})
    all_metrics.append((name, "RMSE", round(rmse, 2), "Regression", "After"))
    all_metrics.append((name, "R2",   round(r2, 4),   "Regression", "After"))
    all_metrics.append((name, "MAE",  round(mae, 2),  "Regression", "After"))
    print(f"✅ {name}: RMSE={rmse:,.0f}, R²={r2:.4f}")

# ── CLASSIFICATION EVALUATION ──
for name in classification_models:
    if name not in models:
        continue
    preds = models[name].transform(test_df)
    acc = mc_eval.evaluate(preds, {mc_eval.metricName: "accuracy"})
    f1  = mc_eval.evaluate(preds, {mc_eval.metricName: "f1"})
    all_metrics.append((name, "Accuracy", round(acc, 4), "Classification", "After"))
    all_metrics.append((name, "F1",       round(f1, 4),  "Classification", "After"))
    # AUC only works for models with probability output (not LinearSVC)
    if name != "Linear_SVC":
        auc = bin_eval.evaluate(preds, {bin_eval.metricName: "areaUnderROC"})
        all_metrics.append((name, "AUC", round(auc, 4), "Classification", "After"))
    print(f"✅ {name}: Accuracy={acc:.4f}, F1={f1:.4f}")

# ── CLUSTERING EVALUATION ──
for name in clustering_models:
    if name not in models:
        continue
    preds = models[name].transform(test_df)
    sil = clu_eval.evaluate(preds)
    all_metrics.append((name, "Silhouette", round(sil, 4), "Clustering", "After"))
    print(f"✅ {name}: Silhouette={sil:.4f}")

# ── ADD "BEFORE TUNING" ROWS (from Notebook 3/4 logged values) ──
# These are the baseline scores BEFORE CrossValidator tuning.
# Replace these with your ACTUAL before-tuning values from MLflow.
before_tuning = [
    ("Linear_Regression",   "RMSE", 42100.0, "Regression",     "Before"),
    ("Decision_Tree_Reg",   "RMSE", 38500.0, "Regression",     "Before"),
    ("Random_Forest_Reg",   "RMSE", 35200.0, "Regression",     "Before"),
    ("GBT_Reg",             "RMSE", 34800.0, "Regression",     "Before"),
    ("Elastic_Net",         "RMSE", 42500.0, "Regression",     "Before"),
    ("Logistic_Regression", "AUC",  0.865,   "Classification", "Before"),
    ("Decision_Tree_Cls",   "AUC",  0.820,   "Classification", "Before"),
    ("Random_Forest_Cls",   "AUC",  0.878,   "Classification", "Before"),
    ("GBT_Cls",             "AUC",  0.882,   "Classification", "Before"),
]
all_metrics.extend(before_tuning)

# ── CREATE SPARK DATAFRAME AND EXPORT ──
metrics_df = spark.createDataFrame(
    all_metrics,
    ["model", "metric", "value", "category", "tuned"]
)
metrics_df.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/workspace/default/fema_claims_project/gold/tableau_metrics")

print(f"\n✅ tableau_metrics.csv exported: {metrics_df.count()} rows")
metrics_df.show(30, truncate=False)


✅ tableau_metrics.csv exported: 9 rows
+-------------------+------+-------+--------------+------+
|model              |metric|value  |category      |tuned |
+-------------------+------+-------+--------------+------+
|Linear_Regression  |RMSE  |42100.0|Regression    |Before|
|Decision_Tree_Reg  |RMSE  |38500.0|Regression    |Before|
|Random_Forest_Reg  |RMSE  |35200.0|Regression    |Before|
|GBT_Reg            |RMSE  |34800.0|Regression    |Before|
|Elastic_Net        |RMSE  |42500.0|Regression    |Before|
|Logistic_Regression|AUC   |0.865  |Classification|Before|
|Decision_Tree_Cls  |AUC   |0.82   |Classification|Before|
|Random_Forest_Cls  |AUC   |0.878  |Classification|Before|
|GBT_Cls            |AUC   |0.882  |Classification|Before|
+-------------------+------+-------+--------------+------+



In [0]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6: DATA QUALITY → tableau_quality.csv
# Columns: column_name, nulls_before, nulls_after, data_type, layer
# ═══════════════════════════════════════════════════════════════════

# ── Load the RAW Bronze data to count nulls BEFORE cleaning ──
raw_df = spark.read.parquet("/Volumes/workspace/default/fema_claims_project/fema_raw.parquet")

# ── Count nulls in key columns BEFORE cleaning ──
key_columns = {
    # column_name: (data_type, medallion_layer)
    "amountPaidOnBuildingClaim":          ("double",  "Silver"),
    "amountPaidOnContentsClaim":          ("double",  "Silver"),
    "totalBuildingInsuranceCoverage":     ("double",  "Silver"),
    "numberOfFloorsInTheInsuredBuilding": ("integer", "Silver"),
    "occupancyType":                     ("integer", "Silver"),
    "floodZone":                         ("string",  "Silver"),
    "state":                             ("string",  "Bronze"),
    "yearOfLoss":                        ("integer", "Bronze"),
    # Engineered features (created in Silver, always 0 nulls)
    "high_payout":                       ("integer", "Gold"),
    "claim_ratio":                       ("double",  "Gold"),
    "flood_risk_score":                  ("double",  "Gold"),
}

quality_rows = []
for col_name, (dtype, layer) in key_columns.items():
    # Count nulls BEFORE (from raw Bronze data)
    if col_name in raw_df.columns:
        nulls_before = raw_df.filter(
            F.col(col_name).isNull()
        ).count()
    else:
        nulls_before = 0  # engineered columns don't exist in Bronze

    # Count nulls AFTER (from cleaned Silver data — should all be 0)
    if col_name in df.columns:
        nulls_after = df.filter(F.col(col_name).isNull()).count()
    else:
        nulls_after = 0

    quality_rows.append((col_name, nulls_before, nulls_after, dtype, layer))
    print(f"  {col_name}: {nulls_before:,} → {nulls_after}")

# ── Export ──
quality_df = spark.createDataFrame(
    quality_rows,
    ["column_name", "nulls_before", "nulls_after", "data_type", "layer"]
)
quality_df.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/workspace/default/fema_claims_project/gold/tableau_quality")

print(f"\n✅ tableau_quality.csv exported: {quality_df.count()} rows")

  amountPaidOnBuildingClaim: 566,225 → 0
  amountPaidOnContentsClaim: 566,225 → 0
  totalBuildingInsuranceCoverage: 4 → 0
  numberOfFloorsInTheInsuredBuilding: 17,248 → 0
  occupancyType: 572 → 512
  floodZone: 0 → 0
  state: 0 → 0
  yearOfLoss: 0 → 0
  high_payout: 0 → 0
  claim_ratio: 0 → 0
  flood_risk_score: 0 → 0

✅ tableau_quality.csv exported: 11 rows


In [0]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5: BUSINESS SUMMARY → tableau_business.csv
# Columns: state, decade, avg_payout, total_claims, avg_coverage, pct_coastal
# Groups 2.7M rows into ~500 rows (one per state-decade pair)
# ═══════════════════════════════════════════════════════════════════

gold_business = df.groupBy("state", "decade").agg(
    F.round(F.avg("amountPaidOnBuildingClaim"), 2).alias("avg_payout"),
    F.count("*").alias("total_claims"),
    F.round(F.avg("totalBuildingInsuranceCoverage"), 2).alias("avg_coverage"),
    F.round(F.avg(F.col("is_coastal").cast("double")), 2).alias("pct_coastal")
)

# ── Sort by avg_payout descending so Tableau opens with top states first ──
gold_business = gold_business.orderBy(F.desc("avg_payout"))

# ── Export ──
gold_business.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/workspace/default/fema_claims_project/gold/tableau_business")

print(f"✅ tableau_business.csv exported: {gold_business.count()} rows")
gold_business.show(10, truncate=False)

✅ tableau_business.csv exported: 335 rows
+-----+------+----------+------------+------------+-----------+
|state|decade|avg_payout|total_claims|avg_coverage|pct_coastal|
+-----+------+----------+------------+------------+-----------+
|FL   |2020  |82094.11  |148122      |540597.82   |1.0        |
|MS   |2000  |74541.53  |26964       |143115.22   |1.0        |
|TX   |2010  |62436.05  |142920      |223075.28   |1.0        |
|SD   |2020  |49542.31  |109         |188521.1    |0.0        |
|VT   |2020  |49393.16  |928         |225589.66   |0.0        |
|NY   |2010  |49228.33  |86300       |259195.18   |1.0        |
|LA   |2000  |48835.42  |233372      |150118.77   |1.0        |
|LA   |2010  |48237.15  |63090       |183669.72   |1.0        |
|HI   |2020  |46281.6   |432         |1288268.06  |0.0        |
|NC   |2020  |44524.6   |6881        |267842.65   |1.0        |
+-----+------+----------+------------+------------+-----------+
only showing top 10 rows


In [0]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6: DATA QUALITY → tableau_quality.csv
# Columns: column_name, nulls_before, nulls_after, data_type, layer
# ═══════════════════════════════════════════════════════════════════

# ── Load the RAW Bronze data to count nulls BEFORE cleaning ──
raw_df = spark.read.parquet("/Volumes/workspace/default/fema_claims_project/fema_raw.parquet")

# ── Count nulls in key columns BEFORE cleaning ──
key_columns = {
    # column_name: (data_type, medallion_layer)
    "amountPaidOnBuildingClaim":          ("double",  "Silver"),
    "amountPaidOnContentsClaim":          ("double",  "Silver"),
    "totalBuildingInsuranceCoverage":     ("double",  "Silver"),
    "numberOfFloorsInTheInsuredBuilding": ("integer", "Silver"),
    "occupancyType":                     ("integer", "Silver"),
    "floodZone":                         ("string",  "Silver"),
    "state":                             ("string",  "Bronze"),
    "yearOfLoss":                        ("integer", "Bronze"),
    # Engineered features (created in Silver, always 0 nulls)
    "high_payout":                       ("integer", "Gold"),
    "claim_ratio":                       ("double",  "Gold"),
    "flood_risk_score":                  ("double",  "Gold"),
}

quality_rows = []
for col_name, (dtype, layer) in key_columns.items():
    # Count nulls BEFORE (from raw Bronze data)
    if col_name in raw_df.columns:
        if dtype == "string":
            nulls_before = raw_df.filter(
                F.col(col_name).isNull() | (F.col(col_name) == "")
            ).count()
        else:
            nulls_before = raw_df.filter(
                F.col(col_name).isNull()
            ).count()
    else:
        nulls_before = 0  # engineered columns don't exist in Bronze

    # Count nulls AFTER (from cleaned Silver data — should all be 0)
    if col_name in df.columns:
        nulls_after = df.filter(F.col(col_name).isNull()).count()
    else:
        nulls_after = 0

    quality_rows.append((col_name, nulls_before, nulls_after, dtype, layer))
    print(f"  {col_name}: {nulls_before:,} → {nulls_after}")

# ── Export ──
quality_df = spark.createDataFrame(
    quality_rows,
    ["column_name", "nulls_before", "nulls_after", "data_type", "layer"]
)
quality_df.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/workspace/default/fema_claims_project/gold/tableau_quality")

print(f"\n✅ tableau_quality.csv exported: {quality_df.count()} rows")

  amountPaidOnBuildingClaim: 566,225 → 0
  amountPaidOnContentsClaim: 566,225 → 0
  totalBuildingInsuranceCoverage: 4 → 0
  numberOfFloorsInTheInsuredBuilding: 17,248 → 0
  occupancyType: 572 → 512
  floodZone: 0 → 0
  state: 0 → 0
  yearOfLoss: 0 → 0
  high_payout: 0 → 0
  claim_ratio: 0 → 0
  flood_risk_score: 0 → 0

✅ tableau_quality.csv exported: 11 rows


In [0]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7: CLUSTER PROFILES → tableau_clusters.csv
# FIX 1: models dict was deleted after evaluation — refit KMeans here
# FIX 2: CSV path was missing leading / on Volumes
# FIX 3: df was never defined — load from Gold layer explicitly
# ═══════════════════════════════════════════════════════════════════
from pyspark.ml.clustering import KMeans
from pyspark.sql import functions as F

# ── FIX 3: Load the Gold layer data explicitly ──────────────────────
# This cell is standalone — df is not carried over from prior cells
df = spark.read.parquet("/Volumes/workspace/default/fema_claims_project/train_prepared/")
#print(f"✅ Gold data loaded: {df.count():,} rows")

CLU_K    = 3
CLU_SEED = 42

clu_sample = df.sample(fraction=0.01, seed=CLU_SEED).repartition(4)
#clu_sample.count()
#print(f"✅ Clustering sample: {clu_sample.count():,} rows")

kmeans = KMeans(
    featuresCol="features",
    predictionCol="prediction",
    k=CLU_K,
    seed=CLU_SEED,
    maxIter=20
)
kmeans_model = kmeans.fit(clu_sample)
print("✅ KMeans refitted")

# ── Apply model to full Gold dataset for cluster profiles ───────────
kmeans_preds = kmeans_model.transform(df)

# ── Aggregate by cluster ────────────────────────────────────────────
# FIX: Guard against missing columns — claim_ratio and is_coastal
# may not exist if Silver engineering was skipped. We check first.
available_cols = df.columns

agg_exprs = [
    F.count("*").alias("count"),
    F.round(F.avg("amountPaidOnBuildingClaim"), 2).alias("avg_payout"),
]

if "claim_ratio" in available_cols:
    agg_exprs.append(F.round(F.avg("claim_ratio"), 4).alias("avg_ratio"))
else:
    agg_exprs.append(F.lit(None).cast("double").alias("avg_ratio"))
    print("⚠️  claim_ratio column not found — avg_ratio will be null")

if "is_coastal" in available_cols:
    agg_exprs.append(
        F.round(F.avg(F.col("is_coastal").cast("double")), 2).alias("pct_coastal")
    )
else:
    agg_exprs.append(F.lit(None).cast("double").alias("pct_coastal"))
    print("⚠️  is_coastal column not found — pct_coastal will be null")

gold_clusters = (
    kmeans_preds
    .groupBy("prediction")
    .agg(*agg_exprs)
    .withColumnRenamed("prediction", "cluster")
    .withColumn("cluster", F.col("cluster").cast("int"))
    .orderBy("cluster")
)

CLUSTER_CSV_PATH = "/Volumes/workspace/default/fema_claims_project/gold/tableau_clusters"

gold_clusters.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(CLUSTER_CSV_PATH)

print(f"✅ tableau_clusters.csv exported to: {CLUSTER_CSV_PATH}")
gold_clusters.show(truncate=False)


✅ KMeans refitted
✅ tableau_clusters.csv exported to: /Volumes/workspace/default/fema_claims_project/gold/tableau_clusters
+-------+------+----------+---------+-----------+
|cluster|count |avg_payout|avg_ratio|pct_coastal|
+-------+------+----------+---------+-----------+
|0      |614489|7989.02   |0.153    |1.0        |
|1      |773668|52510.09  |0.2831   |0.99       |
|2      |461220|11605.34  |0.172    |0.0        |
+-------+------+----------+---------+-----------+



In [0]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8: SCALABILITY ANALYSIS → tableau_scalability.csv
# Columns: data_pct, rows, pyspark_time_s, sklearn_time_s, throughput_rows_per_s
# Tests: train Random Forest at 10%, 25%, 50%, 75%, 100% of data
# Also runs sklearn on 10% as single-node comparison
# ═══════════════════════════════════════════════════════════════════

from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.feature import VectorAssembler, StandardScaler
import os
os.makedirs('/Volumes/workspace/default/fema_claims_project/gold/tableau_scalability', exist_ok=True)

# ── Feature columns (same as training notebooks) ──
feature_cols = [
    "amountPaidOnContentsClaim", "totalBuildingInsuranceCoverage",
    "numberOfFloorsInTheInsuredBuilding",
    "claim_ratio", "claim_age", "is_coastal",
    "has_contents_claim", "has_icc", "decade",
    "coverage_tier", "payout_per_floor", "flood_risk_score"
]
label = "amountPaidOnBuildingClaim"

# ── Simple pipeline for timing (no MLflow overhead) ──
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw",
                            handleInvalid="skip")
scaler = StandardScaler(inputCol="features_raw", outputCol="features_scaled",
                        withMean=True, withStd=True)
rf = RandomForestRegressor(featuresCol="features_scaled", labelCol=label,
                           numTrees=50, maxDepth=10, seed=42)

# ── Weak Scaling: Train at different data sizes ──
fractions = [0.10, 0.25, 0.50, 0.75, 1.00]
total_rows = df.count()
scaling_results = []

for frac in fractions:
    # Sample the data
    if frac < 1.0:
        sample_df = df.sample(frac, seed=42)
    else:
        sample_df = df

    sample_train, sample_test = sample_df.randomSplit([0.8, 0.2], seed=42)
    n_rows = sample_df.count()

    # Time PySpark training
    t0 = time.time()
    from pyspark.ml import Pipeline
    pipe = Pipeline(stages=[assembler, scaler, rf])
    pipe.fit(sample_train)
    pyspark_time = round(time.time() - t0, 1)

    throughput = int(n_rows / pyspark_time) if pyspark_time > 0 else 0

    # sklearn can only run on 10% (memory limit on Community Edition)
    sklearn_time = 0
    if frac <= 0.10:
        try:
            from sklearn.ensemble import RandomForestRegressor as skRF
            pdf = sample_train.select(feature_cols + [label]).toPandas()
            X = pdf[feature_cols].values
            y = pdf[label].values
            t1 = time.time()
            skRF(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1).fit(X, y)
            sklearn_time = round(time.time() - t1, 1)
        except Exception as e:
            print(f"sklearn failed at {frac*100}%: {e}")

    scaling_results.append({
        "data_pct": int(frac * 100),
        "rows": n_rows,
        "pyspark_time_s": pyspark_time,
        "sklearn_time_s": sklearn_time,
        "throughput_rows_per_s": throughput
    })

# ── Export results ──
results_pdf = pd.DataFrame(scaling_results)
results_pdf.to_csv("/Volumes/workspace/default/fema_claims_project/gold/tableau_scalability/scaling_results.csv", index=False)

print(f"✅ tableau_scalability.csv exported: {results_pdf.shape[0]} rows")
display(results_pdf.head())

✅ tableau_scalability.csv exported: 5 rows


data_pct,rows,pyspark_time_s,sklearn_time_s,throughput_rows_per_s
10,186010,17.2,15.0,10814
25,462680,24.0,0.0,19278
50,925378,37.0,0.0,25010
75,1387656,40.0,0.0,34691
100,1849377,51.4,0.0,35980


In [0]:
# ═══════════════════════════════════════════════════════════════════
# CELL 9: VERIFY ALL EXPORTS + DOWNLOAD INSTRUCTIONS
# ═══════════════════════════════════════════════════════════════════

# ── Unpersist cached data to free memory ──
#test_df.unpersist()

# ── List all Gold exports ──
gold_exports = [
    "tableau_business", "tableau_metrics", "tableau_features",
    "tableau_scalability", "tableau_quality", "tableau_clusters"
]

print("═══ GOLD LAYER TABLEAU EXPORTS ═══")
print()
for name in gold_exports:
    path = f"/Volumes/workspace/default/fema_claims_project/gold/{name}"
    try:
        files = dbutils.fs.ls(path)
        csv_files = [f for f in files if f.name.endswith(".csv")]
        if csv_files:
            size_kb = csv_files[0].size / 1024
            print(f"  ✅ {name}.csv — {size_kb:.1f} KB")
        else:
            print(f"  ✅ {name}/ — folder exists")
    except:
        print(f"  ❌ {name} — NOT FOUND")

print()
print("═══ HOW TO DOWNLOAD ═══")
print("Method 1: Catalog → DBFS → data → gold → click each folder")
print("         → download part-00000-xxxxx.csv → rename it")
print()
print("Method 2: Run the cell below to copy to FileStore for direct URL download")

═══ GOLD LAYER TABLEAU EXPORTS ═══

  ✅ tableau_business.csv — 11.3 KB
  ✅ tableau_metrics.csv — 0.4 KB
  ❌ tableau_features — NOT FOUND
  ✅ tableau_scalability.csv — 0.2 KB
  ✅ tableau_quality.csv — 0.5 KB
  ✅ tableau_clusters.csv — 0.1 KB

═══ HOW TO DOWNLOAD ═══
Method 1: Catalog → DBFS → data → gold → click each folder
         → download part-00000-xxxxx.csv → rename it

Method 2: Run the cell below to copy to FileStore for direct URL download
